In [ ]:
import sys, os, gc
sys.path.insert(0, os.path.join('..'))   # project root on path

import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import wrds
import polars as pl
import polars.selectors as cs
import pyarrow
import config

from src.utils import generate_batches, train_val_test_split, split_batch
from src.models.lightgbm import train_lgb
from sklearn.metrics import root_mean_squared_error, r2_score

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.float_format', '{:.4f}'.format)

---
# Generate Batch and Merge

Generating the random sampled batch and merging the other dataset on the batch. 

We merge other dataset with the daily dataset only at the batch level in order to largely decrease the RAM usage duriung the operation considering the size of the daily dataframe with around 6700 stocks.

In [ ]:
"""

Read the features.parquet file and make the batch/split of the data

"""

features = pd.read_parquet(config.FEATURES_PATH_CLEAN).dropna()
cz = pd.read_parquet(config.CZ_PATH_CLEAN)
vix = pd.read_parquet(config.VIX_PATH_CLEAN)

# 1. Merge VIX with CZ
df_tmp = pd.merge(cz, vix, on='date')

print(f'Number of unique PERMNO in dataset : {features['PERMNO'].nunique()}')

split_batch_dict = split_batch(features, 'PERMNO', 'date', df_to_join=[df_tmp], batch_number=config.BATCH_NUMBER, batch_size=config.BATCH_SIZE, overlap=False)

del features, cz, vix, df_tmp
gc.collect()

In [5]:
""""

Unique batch test

"""
from sklearn.linear_model import LassoCV, Lasso, lasso_path


# 1. Initiate LASSO
model = LassoCV(cv=5, n_jobs=-1)


# 2. Set training and validation set
TARGET = 'target'

y_train = split_batch_dict['batch_1']['train'][TARGET].reset_index(drop=True)
X_train = split_batch_dict['batch_1']['train'].drop([TARGET, 'ret', 'mkt_ret'], axis=1).reset_index(drop=True)

y_val = split_batch_dict['batch_1']['validation'][TARGET].reset_index(drop=True)
X_val = split_batch_dict['batch_1']['validation'].drop([TARGET, 'ret', 'mkt_ret'], axis=1).reset_index(drop=True)


# Trouver alpha sur un échantillon représentatif
sample_idx = np.random.choice(len(X_train), size=50_000, replace=False)
cv_model = LassoCV(cv=5, n_jobs=1)
cv_model.fit(X_train.iloc[sample_idx], y_train.iloc[sample_idx])

best_alpha = cv_model.alpha_
print(f"Alpha optimal : {best_alpha:.6f}")

model = Lasso(alpha=best_alpha)
model.fit(X_train, y_train)

print(f"R² train : {r2_score(y_train, model.predict(X_train)):.4f}")
print(f"R² val   : {r2_score(y_val, model.predict(X_val)):.4f}")


Alpha optimal : 0.000011
R² train : 0.0054
R² val   : 0.0009
